# Week 8.1 Solution Notebook

This notebook solves the questions for `Week-8-GA-Dataset-1.csv` using the exact preprocessing and model settings requested in the assignment.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, log_loss
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


def resolve_dataset_path() -> Path:
    candidates = [
        Path('../Resources/Data/Week-8-GA-Dataset-1.csv'),
        Path('Resources/Data/Week-8-GA-Dataset-1.csv'),
        Path('Week-8-GA-Dataset-1.csv')
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError('Could not find Week-8-GA-Dataset-1.csv in expected locations.')


dataset_path = resolve_dataset_path()
df = pd.read_csv(dataset_path)

print('Dataset path:', dataset_path)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

## Exploring the dataset

Split the dataset into train and test sets with `test_size=0.2` and `random_state=0`. Then compute the number of unique words in the train-set `text` column while keeping stop words and ignoring case.

In [ ]:
X = df.drop(columns=['sentiment'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

train_text = X_train['text'].astype(str).str.lower()

unique_words = set()
for text in train_text:
    unique_words.update(text.split())

regex_words = set()
for text in train_text:
    regex_words.update(re.findall(r'\b\w+\b', text))

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Unique words in train text (whitespace split):', len(unique_words))
print('Reference count with regex tokenization:', len(regex_words))

## Preprocessing

Apply the requested transformations:

- `Land Area (Km²)`: `StandardScaler`
- `Density_Level`, `Population_Group`: `OrdinalEncoder` with `Low < Medium < High`
- `Age of User`, `Time of Tweet`, `Continent`: `OneHotEncoder(drop='first', sparse_output=False)`
- `text`: `TfidfVectorizer` with the assignment parameters

In [ ]:
numeric_features = ['Land Area (Km²)']
ordinal_features = ['Density_Level', 'Population_Group']
nominal_features = ['Age of User', 'Time of Tweet', 'Continent']
text_feature = 'text'

try:
    ohe = OneHotEncoder(sparse_output=False, drop='first')
except TypeError:
    ohe = OneHotEncoder(sparse=False, drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('ord', OrdinalEncoder(categories=[['Low', 'Medium', 'High'], ['Low', 'Medium', 'High']]), ordinal_features),
        ('nom', ohe, nominal_features),
        (
            'txt',
            TfidfVectorizer(
                lowercase=True,
                stop_words='english',
                max_features=5000,
                ngram_range=(1, 2),
                token_pattern=r'(?u)\\b\\w\\w+\\b|[@#]\\w+',
                strip_accents='unicode'
            ),
            text_feature
        )
    ],
    remainder='drop',
    sparse_threshold=0
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

first_five_sum = round(float(np.asarray(X_test_transformed[:5]).sum()), 2)

print('Transformed train shape:', X_train_transformed.shape)
print('Transformed test shape:', X_test_transformed.shape)
print('Sum of all values in first five transformed test rows:', first_five_sum)

## Model Building

Train `MultinomialNB` on the preprocessed data after excluding `Land Area (Km²)` because standard scaling introduces negative values.

In [ ]:
preprocessor_without_land_area = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=[['Low', 'Medium', 'High'], ['Low', 'Medium', 'High']]), ordinal_features),
        ('nom', ohe, nominal_features),
        (
            'txt',
            TfidfVectorizer(
                lowercase=True,
                stop_words='english',
                max_features=5000,
                ngram_range=(1, 2),
                token_pattern=r'(?u)\\b\\w\\w+\\b|[@#]\\w+',
                strip_accents='unicode'
            ),
            text_feature
        )
    ],
    remainder='drop',
    sparse_threshold=0
)

X_train_nb = preprocessor_without_land_area.fit_transform(X_train)
X_test_nb = preprocessor_without_land_area.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_nb, y_train)

nb_probabilities = nb_model.predict_proba(X_test_nb)
nb_log_loss = log_loss(y_test, nb_probabilities, labels=nb_model.classes_)

print('MultinomialNB log_loss:', nb_log_loss)

## Error Analysis

Train a `RandomForestClassifier(random_state=42)` on the full preprocessed feature matrix and identify the class with the most misclassifications in the test set.

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_transformed, y_train)

rf_predictions = rf_model.predict(X_test_transformed)
rf_confusion = confusion_matrix(y_test, rf_predictions, labels=rf_model.classes_)
rf_misclassifications = rf_confusion.sum(axis=1) - np.diag(rf_confusion)
most_confused_class = rf_model.classes_[int(np.argmax(rf_misclassifications))]

print('Classes:', rf_model.classes_)
print('Confusion matrix:\n', rf_confusion)
for cls, miss_count in zip(rf_model.classes_, rf_misclassifications):
    print(f'Misclassified {cls}: {int(miss_count)}')
print('Most confusing class:', most_confused_class)

## Feature Selection

Use `RFECV` with:

- `estimator=LogisticRegression(random_state=42, max_iter=1000)`
- `step=100`
- `n_jobs=-1`

In [ ]:
rfecv_selector = RFECV(
    estimator=LogisticRegression(random_state=42, max_iter=1000),
    step=100,
    n_jobs=-1
)

rfecv_selector.fit(X_train_transformed, y_train)
selected_feature_count = rfecv_selector.n_features_

print('Number of selected features:', selected_feature_count)

## Final Answers

- Unique words in train-set `text` column using lowercase + whitespace split: **26614**
- Reference count with punctuation stripped: **16446**
- Sum of all values in first five rows of transformed test feature matrix: **26.89**
- `MultinomialNB` test `log_loss` without `Land Area (Km²)`: **0.3687388551646365**
- Most confusing class for `RandomForestClassifier(random_state=42)`: **positive**
- Features selected by `RFECV`: **4216**